In [1]:
import csv
import colorsys
import random
import os
import string
import json
from pprint import pprint
from pipeline import GeoGridPipeline

## Grid representation

In [ ]:
variety2Wikidata = dict()
with open("variety2Wikidata.tsv", "r") as f:
	# Location is unique. Set it as the key to the dictionary
	for i in f.read().splitlines()[1:]:
		country = i.split("\t")[0]
		language = i.split("\t")[1]
		location = i.split("\t")[2]
		wikidata = i.split("\t")[3]
		if location not in variety2Wikidata:
			variety2Wikidata.update({location: {"country": country, "language": language, "Wikidata": wikidata}})
		else:
			print("Missing location: ", location)
			
p = GeoGridPipeline(grid_size=0.05, cache_file="regions_cache.tsv")

In [3]:
# Stage 1 (only once):
# qids = [(v["Wikidata"].strip().split("/")[-1], location, v["language"])
#         for location, v in variety2Wikidata.items()]
# p.resolve_and_save(qids, path="regions_cache_new.tsv")#, override_path="regions_cache_overrides.tsv")

In [ ]:
# Stage 2:
p.load_and_build()
p.grid.visualize(extent=[-25, 45, 34, 72])
#p.grid.visualize(extent=[-25, 45, 34, 72], cell_colors=p.cell_colors)

p.grid.visualize(extent=[-25, 70, 5, 72])

In [ ]:
import csv
import colorsys
import random
import os
import string
import json
from pprint import pprint
from pipeline import GeoGridPipeline

variety2Wikidata = dict()
with open("variety2Wikidata.tsv", "r") as f:
	# Location is unique. Set it as the key to the dictionary
	for i in f.read().splitlines()[1:]:
		country = i.split("\t")[0]
		language = i.split("\t")[1]
		location = i.split("\t")[2]
		wikidata = i.split("\t")[3]
		if location not in variety2Wikidata:
			variety2Wikidata.update({location: {"country": country, "language": language, "Wikidata": wikidata}})
		else:
			print("Missing location: ", location)
			
p = GeoGridPipeline(grid_size=0.05, cache_file="regions_cache.tsv")

CONTINUA = {
	"Italic": ["Italian"],
	# "Arabic": ["Arabic"],
	"North Germanic": ["Danish", "Swedish", "Norwegian", "Faroese", "Ovdalian", "Icelandic"],
	"Caspian": ["Gilaki", "Talysh", "Mazandarani"],
	"Ukrainian": ["Ukrainian"],
	"Kurdic": ["Central Kurdish", "Southern Kurdish", "Hawrami", "Laki Kurdish", "Northern Kurdish"],
	# "Hellenic": ["Greek"],
	"West Germanic": ["Dutch", "German", "Swiss German"],
	"Basque": ["Basque"],
}

p.load_and_build()
p.grid.visualize_continua(
	continua=CONTINUA,
	# extent=[-25, 65, 10, 72],
	save_path="dialect_maps.pdf",
	figsize=(3.75, 2.5)
)

In [ ]:
cache = dict()

with open("regions_cache.tsv", "r") as f:
	cache_content = set(f.read().splitlines())
	for line in cache_content:
		line_cols = line.split("\t")
		cache.update({line_cols[1]: {"qid": line_cols[0], 
							   "country": line_cols[2],
							   "language": line_cols[3],
							   "type": line_cols[4],
								"min_lat": line_cols[5],
								"max_lat": line_cols[6],
								"min_lon": line_cols[7],
								"max_lon": line_cols[8]}})
		
pprint(cache)

### CODET Data (Italian, Arabic, Basque, Kurdish)

In [5]:
def count_character(text):
	# counts number of non-space, non-punctuation characters in a string
	return len(text.translate(str.maketrans('', '', string.punctuation)).replace(' ', ''))

def split_long_text(text, max_len=100, min_len=10):
	"""Split text into chunks, avoiding very short final chunks."""
	chunks = []
	while len(text) > max_len:
		split_pos = text.rfind(' ', 0, max_len)
		if split_pos == -1:
			split_pos = max_len
		chunks.append(text[:split_pos].strip())
		text = text[split_pos:].strip()
	if text:
		if len(text) >= min_len and len(text.split()) >= 2:
			chunks.append(text)
		elif chunks:
			chunks[-1] = chunks[-1] + ' ' + text
	return chunks

def preprocess_data(samples, max_len=100, min_len=10):
	new_samples = []
	for sample in samples:
		if len(sample) >= min_len and len(sample.split()) >= 2:
			if len(sample) > max_len:
				new_samples.extend(split_long_text(sample, max_len, min_len))
			else:
				new_samples.append(sample)
	return new_samples

In [ ]:
# Italian, Arabic, Basque, Kurdish
file_to_dialect = {}
dialect_data = {}

lang_files = {
	"Italian": "datasets/Italian/",
	# "Arabic": "datasets/Arabic/", # skip for now
	"Basque": "datasets/Basque/",
	"Central Kurdish": "datasets/Kurdish/"
	}

for language in lang_files:
	dialect_data.update({language: {}})
	files = os.listdir(lang_files[language])

	for f in files:
		if language == "Italian":
			dialect = f.replace(".ita", "").replace("_", " ")
			# Remove trailing numbers
			dialect = dialect.rstrip("0123456789").strip()
		if language == "Arabic":
			dialect = f.replace(".arb", "")
		if language == "Basque":
			dialect = f.replace(".eus", "")
		if language == "Central Kurdish":
			dialect = f.replace(".ckb", "")
		
		file_to_dialect[f] = dialect
		
		if dialect in cache:
			with open(lang_files[language] + f, "r") as dialect_f:
				if dialect not in dialect_data[language]:
					dialect_data[language].update({dialect: {
						"sentences": [i for i in set(dialect_f.read().splitlines()) if not(count_character(i) < 5 and i.count(' ') < 2)],
						"meta": cache[dialect]}})
				else:
					dialect_data[language][dialect]["sentences"].extend([i for i in set(dialect_f.read().splitlines()) if not(count_character(i) < 5 and i.count(' ') < 2)])
				
		else:
			print("Missing location: ", dialect, f)

### Greek

In [ ]:
language = "Greek"
if language not in dialect_data:
	dialect_data[language] = {}

folder_path = "datasets/Greek-GRDD/"
variety2file = {
	# "Pontic": "pontic.txt",
	"Cyprus": "cypriot.txt",
	"Crete": "cretan.txt",
	"Thessaly": "nothern.txt",
}

for var in variety2file:
	with open(folder_path + variety2file[var], "r") as f:
		var_content = f.read().splitlines()
		new_var_content = list()
		if var == "Cretan":
			for i in var_content:
				if not i.startswith('---'):
					new_var_content.extend(i.split(". "))
			var_content = new_var_content
		data = list(set(preprocess_data(var_content)))
		print(f"{var}: # Lines (original): {len(var_content)} / # Lines (preprocessed): {len(data)}")

		if var not in cache:
			print(f"  Warning: '{var}' not in cache, skipping")
			continue

		if var not in dialect_data[language]:
			dialect_data[language][var] = {
				"sentences": data,
				"meta": cache[var],
			}
		else:
			dialect_data[language][var]["sentences"].extend(data)
			dialect_data[language][var]["sentences"] = list(
				set(dialect_data[language][var]["sentences"])
			)

# pprint(dialect_data, width=200)

## German

In [ ]:
# Download https://github.com/mainlp/betthupferl and extract Betthupferl-data_processed/pure_text/ in the dataset folder
language = "German"
dialect_data[language] = {}

folder_path = "datasets/Betthupferl-data_processed/pure_text/"
variety2file = {
	"oberfranken": "oberfranken_gold_dial.txt",
	"mittelfranken": "mittelfranken_gold_dial.txt",
	"niederbayern": "niederbayern_gold_dial.txt",
	"oberbayern": "oberbayern_gold_dial.txt",
	"oberpfalz": "oberpfalz_gold_dial.txt",
	"schwaben": "schwaben_gold_dial.txt",
	"unterfranken": "unterfranken_gold_dial.txt",
}

for var in variety2file:
	with open(folder_path + variety2file[var], "r") as f:
		var_content = f.read().splitlines()

	# Strip leading/trailing quotation marks
	var_content = [line.strip("\"'") for line in var_content]
	
	data = list(set(preprocess_data(var_content)))
	print(f"{var}: # Lines (original): {len(var_content)} / # Lines (preprocessed): {len(data)}")

	if var not in cache:
		print(f"  Warning: '{var}' not in cache, skipping")
		continue

	if var not in dialect_data[language]:
		dialect_data[language][var] = {
			"sentences": data,
			"meta": cache[var],
		}
	else:
		dialect_data[language][var]["sentences"].extend(data)
		dialect_data[language][var]["sentences"] = list(
			set(dialect_data[language][var]["sentences"])
		)

## Scandinavian

Source: https://github.com/textlab/spoken_norwegian_resources/tree/master/transcriptions/NDC/txt/ndc_with_informant_codes

In [ ]:
import os
import re
import json
from geopy.geocoders import Nominatim
import fasttext
import random

from SPARQLWrapper import SPARQLWrapper, JSON
from rapidfuzz import fuzz
from tqdm import tqdm

In [ ]:
lang_dir = {
	"Danish": "spoken_norwegian_resources/files/danish",
	"Faroese": "spoken_norwegian_resources/files/faroese",
	"Icelandic": "spoken_norwegian_resources/files/icelandic",
	"Norwegian": "spoken_norwegian_resources/files/norwegian",
	"Ovdalian": "spoken_norwegian_resources/files/ovdalian",
	"Swedish": "spoken_norwegian_resources/files/swedish"
}

def read_language_files():	
	# Initialize the result dictionary
	language_data = {}
	
	# Process each language
	for language, directory in lang_dir.items():

		language_data[language] = {}
		
		# Get all files in the directory
		files = os.listdir(directory)
		
		for filename in files:
			file_path = os.path.join(directory, filename)
			
			# Skip directories, only process files
			if os.path.isfile(file_path):
				# Extract dialect from filename (part before first underscore)
				# Example: aarhus1_interview_klippet_K_HJ_lh -> aarhus1
				dialect = filename.split('_')[0]

				# remove metadata from the dialect name
				dialect_label = dialect.capitalize()
				if language == "Swedish":
					dialect_label = re.sub(r'[\d-]+$', '', dialect_label)
					dialect_label = re.sub(r'[a-z]{2}$', '', dialect_label)
					dialect_label = re.sub(r'[A-Z]+$', '', dialect_label)
					if dialect_label == "St":
						dialect_label = "St Anna"
					elif dialect_label == "Leksando":
						dialect_label = "Leksand"

				elif language == "Danish":
					dialect_label = re.sub(r'[\d-]+$', '', dialect_label)
				
				elif language == "Norwegian":
					dialect_label = re.sub(r'[A-Z]+$', '', dialect).capitalize()

				# Initialize dialect if not exists
				if dialect_label not in language_data[language]:
					language_data[language][dialect_label] = []
				
				# Read file content
				# dialect_sentences = list()
				with open(file_path, 'r', encoding='utf-8') as f:
					text_lines = f.readlines()
					
					# remove sentences from the interviewer and remove metadata from the dialect speaker
					for line in text_lines:
						if line.strip():
							# skip interviewer's utterance as it may represent non-dialectal data
							first_token = line.split(" ")[0].lower()
							if first_token != "interviewer" and first_token != "int" and first_token != "ls":
								# remove identity of the speaker
								if dialect_label.lower() in first_token:
									line = " ".join(line.split(" ")[1:])
								
								line = line.strip()
								line = line.replace("* ", "", 1)
								
								if len(line) > 2:
									language_data[language][dialect_label].append(line)
					language_data[language][dialect_label] = list(set(language_data[language][dialect_label]))
	
	return language_data

data = read_language_files()

with open('datasets/Scandinavian_data.json', 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

In [ ]:
for lang in lang_dir:
    if lang in data:
        dialects = len(data[lang])
        lines_per_dialect = [len(lines) for lines in data[lang].values()]
        total_lines_lang = sum(lines_per_dialect)
        min_lines = min(lines_per_dialect) if lines_per_dialect else 0
        max_lines = max(lines_per_dialect) if lines_per_dialect else 0
        avg_lines = sum(lines_per_dialect) / len(lines_per_dialect) if lines_per_dialect else 0
        
        print(f"{lang:15}: {dialects:3} dialects, {total_lines_lang:6} lines (min: {min_lines:3}, max: {max_lines:4}, avg: {avg_lines:5.1f})")
    else:
        print(f"{lang:15}: NOT FOUND")

# Summary
total_dialects = sum(len(dialects) for dialects in data.values())
total_lines = sum(len(lines) for lang in data.values() for lines in lang.values())
print(f"\nSummary: {len(data)} languages, {total_dialects} dialects, {total_lines} lines")
print(f"Average: {total_dialects/len(data):.1f} dialects/lang, {total_lines/total_dialects:.1f} lines/dialect")


In [ ]:
with open("datasets/Scandinavian_data.json", "r") as f:
	content = json.load(f)
	
	for language in content:
		if language not in dialect_data:
			dialect_data.update({language: {}})
	
		for dialect in content[language]:
			sentences = list()
			if dialect in cache:
				for i in set(content[language][dialect]):
					if count_character(i) > 5 and i.count(' ') > 2:
						sentences.append(" ".join(i.replace("#", " ").split()))

				dialect_data[language].update({dialect: {"sentences": sentences, "meta": cache[dialect]}})
				
			else:
				print("Missing location: ", dialect, f)

### Swiss German

Download https://mtc.ethz.ch/publications/open-source/swiss-dial.html and save it in "datasets" as "sentences_ch_de_transcribed.json"

In [ ]:
key_to_dialect = {
	"ch_ag": "Aargau",
	"ch_be": "Bern",
	"ch_bs": "Basel",
	"ch_gr": "Graubünden",
	"ch_lu": "Luzern",
	"ch_sg": "St. Gallen",
	"ch_vs": "Wallis",
	"ch_zh": "Zürich",
}

language = "Swiss German"
if language not in dialect_data:
	dialect_data[language] = {}

with open("datasets/sentences_ch_de_transcribed.json") as f:
	records = json.load(f)

for row in records:
	for key, dialect in key_to_dialect.items():
		sentence = row.get(key, "").strip()
		sentence = ' '.join(sentence.split())
		if not sentence:
			continue
		if count_character(sentence) <= 10 or sentence.count(' ') <= 2:
			continue
		if dialect not in dialect_data[language]:
			dialect_data[language][dialect] = {
				"sentences": [sentence],
				"meta": cache[dialect],
			}
		else:
			dialect_data[language][dialect]["sentences"].append(sentence)

for dialect in dialect_data[language]:
	dialect_data[language][dialect]["sentences"] = list(
		set(dialect_data[language][dialect]["sentences"])
	)

### Northern Kurdish

In [ ]:
language = "Northern Kurdish"
if language not in dialect_data:
	dialect_data[language] = {}

with open("datasets/Manchester-database-Kurdish-Dialects.csv", newline='') as f:
	reader = csv.DictReader(f, delimiter=',')
	for row in reader:
		dialect = row['location']
		sentence = row['phrase'].strip()

		if dialect not in cache:
			continue

		# Filter short sentences
		if count_character(sentence) <= 10 or sentence.count(' ') <= 2:
			continue

		if dialect not in dialect_data[language]:
			dialect_data[language][dialect] = {
				"sentences": [sentence],
				"meta": cache[dialect],
			}
		else:
			dialect_data[language][dialect]["sentences"].append(sentence)

# Deduplicate sentences
for dialect in dialect_data[language]:
	dialect_data[language][dialect]["sentences"] = list(
		set(dialect_data[language][dialect]["sentences"])
	)

### Dutch

Check "util/extract_sand.py" for the preprocessing script.

In [ ]:
language = "Dutch"
if language not in dialect_data:
	dialect_data[language] = {}

with open("datasets/NL_sand_corpus.tsv", newline='') as f:
	reader = csv.DictReader(f, delimiter='\t')
	for row in reader:
		dialect = row['location_name'].split("/")[0].strip().replace("'", "")
		sentence = row['text'].strip()
		sentence = ' '.join(sentence.split())

		if dialect not in cache:
			print("missing dialect: ", dialect)

		if count_character(sentence) <= 10 or sentence.count(' ') <= 2:
			continue

		if dialect not in dialect_data[language]:
			dialect_data[language][dialect] = {
				"sentences": [sentence],
				"meta": cache[dialect],
			}
		else:
			dialect_data[language][dialect]["sentences"].append(sentence)

# Deduplicate
for dialect in dialect_data[language]:
	dialect_data[language][dialect]["sentences"] = list(
		set(dialect_data[language][dialect]["sentences"])
	)

# pprint(dialect_data, width=200)

### Ukrainian

In [ ]:
language = "Ukrainian"
if language not in dialect_data:
	dialect_data[language] = {}

folder_path = "datasets/Ukrainian/"

for filename in os.listdir(folder_path):
	if not filename.endswith(".txt"):
		continue

	var = filename.replace(".txt", "")

	with open(folder_path + filename, "r") as f:
		var_content = f.read().splitlines()

	data = list(set(preprocess_data(var_content)))
	print(f"{var}: # Lines (original): {len(var_content)} / # Lines (preprocessed): {len(data)}")

	if var not in cache:
		print(f"  Warning: '{var}' not in cache, skipping")
		continue

	if var not in dialect_data[language]:
		dialect_data[language][var] = {
			"sentences": data,
			"meta": cache[var],
		}
	else:
		dialect_data[language][var]["sentences"].extend(data)
		dialect_data[language][var]["sentences"] = list(
			set(dialect_data[language][var]["sentences"])
		)

# pprint(dialect_data[language], width=2000)

### PARME

In [ ]:
folder_path = "datasets/PARME/"

parme_files = {
	"Gilaki": ["en-fa-glk.tsv"],
	"Hawrami": ["en-fa-hac.tsv", "HAC-EN.tsv"],
	"Laki Kurdish": ["en-fa-lki.tsv", "LKI-EN.tsv"],
	"Mazandarani": ["en-fa-mzn.tsv"],
	"Southern Kurdish": ["en-fa-sdh.tsv", "SDH-EN.tsv"],
	"Talysh": ["en-fa-tly.tsv"],
}

for language, filenames in parme_files.items():
	if language not in dialect_data:
		dialect_data[language] = {}

	missing_dialects = set()

	for filename in filenames:
		with open(folder_path + filename, newline='') as f:
			reader = csv.DictReader(f, delimiter='\t')
			headers = reader.fieldnames

			for row in reader:
				dialect = row['county'].split("/")[0].strip()
				sentence = row['translation'].strip()
				sentence = ' '.join(sentence.split())

				if dialect not in cache or dialect == '':
					missing_dialects.add(dialect)
					continue

				if count_character(sentence) <= 10 or sentence.count(' ') <= 2:
					continue

				if dialect not in dialect_data[language]:
					dialect_data[language][dialect] = {
						"sentences": [sentence],
						"meta": cache[dialect],
					}
				else:
					dialect_data[language][dialect]["sentences"].append(sentence)

	if missing_dialects:
		print(f"  WARNING [{language}]: These locations not in cache: {missing_dialects}")

	# Deduplicate
	for dialect in dialect_data[language]:
		dialect_data[language][dialect]["sentences"] = list(
			set(dialect_data[language][dialect]["sentences"])
		)

# Summary
for language in parme_files:
	if language in dialect_data:
		for dialect in dialect_data[language]:
			print(f"{language} / {dialect}: {len(dialect_data[language][dialect]['sentences'])} sentences")
		if not dialect_data[language]:
			print(f"{language}: NO DATA LOADED")

In [14]:
with open("dialect_data.json", "w", encoding="utf-8") as f:
	json.dump(dialect_data, f, ensure_ascii=False, indent=4)

### Corpus stats

In [16]:
with open("dialect_data.json", "r") as f:
	dialect_data = json.load(f)

sources = {
	"Italian": "ASIt~\\citep{pescarini2010database}",
	"Norwegian": "NorDiaSyn~\\citep{johannessen2011nordic}",
	"Swedish": "SweDiaSyn~\\citep{johannessen2011nordic}",
	"Icelandic": "ScanDiaSyn~\\citep{johannessen2011nordic}",
	"Danish": "DanDiaSyn~\\citep{johannessen2011nordic}",
	"Ovdalian": "ScanDiaSyn~\\citep{johannessen2011nordic}",
	"Faroese": "ScanDiaSyn~\\citep{johannessen2011nordic}",
	"Basque": "BASYQUE~\\citep{uria2011basyque}",
	"German": "Betthupferl~\\citep{blaschke-etal-2025-multi}",
	"Swiss German": "SwissDial~\\citep{DBLP:journals/corr/abs-2103-11401}",
	"Greek": "GRDD~\\citep{DBLP:journals/corr/abs-2308-00802}",
	"Northern Kurdish": "Manchester~\\citep{matras2019revisiting}",
	"Central Kurdish": "CORDI~\\citep{DBLP:conf/coling/AhmadiJAA24}",
	"Southern Kurdish": "PARME~\\citep{DBLP:conf/acl/AhmadiSKMFBHHDA25}",
	"Hawrami": "PARME~\\citep{DBLP:conf/acl/AhmadiSKMFBHHDA25}",
	"Laki Kurdish": "PARME~\\citep{DBLP:conf/acl/AhmadiSKMFBHHDA25}",
	"Gilaki": "PARME~\\citep{DBLP:conf/acl/AhmadiSKMFBHHDA25}",
	"Mazandarani": "PARME~\\citep{DBLP:conf/acl/AhmadiSKMFBHHDA25}",
	"Talysh": "PARME~\\citep{DBLP:conf/acl/AhmadiSKMFBHHDA25}",
	"Ukrainian": "~\\citep{frund2026ukrainian}",
	"Dutch": "SAND~\\citep{barbiers2007syntactic}",
}

print(f"{'Language':<20} {'Source':<55} {'# Geo':>6} {'# Samples':>10} {'Avg/Geo':>8} {'Tokens':>10} {'TTR':>8} {'AvgWrd':>8}")
print("-" * 130)

total_geo = 0
total_samples = 0
total_tokens = 0
all_types = set()
all_word_counts = []

for language in sorted(dialect_data.keys()):
	dialects = dialect_data[language]
	num_geolects = len(dialects)
	total_geo += num_geolects

	all_sentences = []
	for dialect in dialects.values():
		all_sentences.extend(dialect["sentences"])

	num_samples = len(all_sentences)
	total_samples += num_samples
	avg_per_geo = num_samples / num_geolects if num_geolects > 0 else 0

	tokens = []
	word_counts = []
	for s in all_sentences:
		words = s.split()
		tokens.extend(words)
		word_counts.append(len(words))

	num_tokens = len(tokens)
	ttr = len(set(tokens)) / num_tokens if num_tokens > 0 else 0
	avg_wrd = sum(word_counts) / len(word_counts) if word_counts else 0

	total_tokens += num_tokens
	all_types.update(tokens)
	all_word_counts.extend(word_counts)

	src = sources.get(language, "")
	print(f"{language:<20} {src:<55} {num_geolects:>6} {num_samples:>10} {avg_per_geo:>8.1f} {num_tokens:>10} {ttr:>8.3f} {avg_wrd:>8.1f}")

print("-" * 130)
avg_geo = total_samples / total_geo if total_geo > 0 else 0
avg_wrd_total = sum(all_word_counts) / len(all_word_counts) if all_word_counts else 0
ttr_total = len(all_types) / total_tokens if total_tokens > 0 else 0
print(f"{'TOTAL':<20} {'':<55} {total_geo:>6} {total_samples:>10} {avg_geo:>8.1f} {total_tokens:>10} {ttr_total:>8.3f} {avg_wrd_total:>8.1f}")

Language             Source                                                   # Geo  # Samples  Avg/Geo     Tokens      TTR   AvgWrd
----------------------------------------------------------------------------------------------------------------------------------
Basque               BASYQUE~\citep{uria2011basyque}                             40       7610    190.2      37508    0.167      4.9
Central Kurdish      CORDI~\citep{DBLP:conf/coling/AhmadiJAA24}                   5     117221  23444.2     789887    0.127      6.7
Danish               DanDiaSyn~\citep{johannessen2011nordic}                     13       7851    603.9     173658    0.055     22.1
Dutch                SAND~\citep{barbiers2007syntactic}                         267      55582    208.2     436259    0.034      7.8
Faroese              ScanDiaSyn~\citep{johannessen2011nordic}                     5       3641    728.2      61522    0.096     16.9
German               Betthupferl~\citep{blaschke-etal-2025-multi}      